# Capability 11 — OpenRouter GLM Copilot

Fallback answers for the U001 question. A live OpenRouter cell is skipped without a key.


In [1]:
import json
import os
from datetime import datetime, timedelta
from decimal import Decimal
from pathlib import Path

import pandas as pd

from telco_digital.application.clock import FixedClock
from telco_digital.application.seed import seed_demo_customers
from telco_digital.config import Settings
from telco_digital.copilot import CopilotService
from telco_digital.decisioning import DecisionEngine
from telco_digital.infrastructure.memory import InMemoryUnitOfWork
from telco_digital.intelligence.behaviour import BehaviourService
from telco_digital.intelligence.churn import ChurnService
from telco_digital.intelligence.event_memory import EventMemoryService
from telco_digital.intelligence.event_memory.uow import UnitOfWorkEventMemoryQueries
from telco_digital.intelligence.features import CustomerFeatures, GraphFeatures
from telco_digital.intelligence.features.service import FeatureGroup
from telco_digital.intelligence.recommendations import PlanRepositoryCatalogue, RecommendationService

ROOT = Path(".")
for folder in ("outputs/tables", "outputs/plots", "artifacts"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
AS_OF = datetime.fromisoformat("2026-08-20T12:00:00+00:00")
QUESTION = "Why is U001 receiving this recommendation?"


In [2]:
async def features_from_uow(uow, customer_ref, as_of):
    customer = await uow.customers.get_by_ref(customer_ref)
    start_90 = as_of - timedelta(days=90)
    usage = [row for row in await uow.usage_events.list_as_of(customer.id, as_of) if row.occurred_at >= start_90]
    return CustomerFeatures(
        customer_id=customer.id,
        customer_ref=customer_ref,
        as_of=as_of,
        computed_at=as_of,
        temporal={
            "usage": FeatureGroup(window_days=30, values={"data_mb_30d": 0, "data_mb_90d": 0, "data_mb_change_ratio": None}),
            "recharge": FeatureGroup(window_days=30, values={"small_recharge_count_30d": 0}),
            "service": FeatureGroup(window_days=90, values={"complaint_count_90d": 0, "open_count": 0}),
        },
        graph=GraphFeatures(available=False, values={}),
        provenance=("in-memory seed",),
    )


class UowFeatures:
    def __init__(self, uow):
        self.uow = uow

    async def calculate(self, customer_ref, as_of):
        return await features_from_uow(self.uow, customer_ref, as_of)


uow = InMemoryUnitOfWork()
await seed_demo_customers(uow, clock=FixedClock(AS_OF))
memory = EventMemoryService(UnitOfWorkEventMemoryQueries(uow))
features = UowFeatures(uow)
engine = DecisionEngine(
    RecommendationService(memory, PlanRepositoryCatalogue(uow.plans)),
    BehaviourService(features, memory),
    ChurnService(features),
)
answer = await CopilotService(engine, Settings()).answer(QUESTION, "U001", AS_OF, destination="SG")
row = pd.DataFrame([{
    "source": answer.source,
    "answer": answer.answer,
    "used_facts": " | ".join(answer.used_facts),
    "mentions_roam_15": "ROAM_15" in answer.answer,
    "mentions_duration_unknown": "duration" in answer.answer.lower() and "unknown" in answer.answer.lower(),
    "mentions_fake_plan": "FAKE_PLAN" in answer.answer,
}])
row.to_json(ROOT / "outputs" / "tables" / "u001_fallback.json", orient="records", indent=2)
metrics = {
    "source": answer.source,
    "mentions_roam_15": bool(row.loc[0, "mentions_roam_15"]),
    "mentions_duration_unknown": bool(row.loc[0, "mentions_duration_unknown"]),
    "mentions_fake_plan": bool(row.loc[0, "mentions_fake_plan"]),
}
(ROOT / "outputs" / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
row


,source,answer,used_facts,mentions_roam_15,mentions_duration_unknown,mentions_fake_plan
0,deterministic_fallback,Question: Why is U001 receiving this recommend...,action=PRESENT_OFFER | target=ROAM_15 | HISTOR...,True,True,False


In [3]:
key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
if not key:
    print("OPENROUTER_API_KEY is not set; skipping the live model cell.")
else:
    live = await CopilotService(engine, Settings(openrouter_api_key=key)).answer(
        QUESTION, "U001", AS_OF, destination="SG"
    )
    print(live.source)
    print(live.answer)


OPENROUTER_API_KEY is not set; skipping the live model cell.
